# YOLOv4-tiny Roboflow Playground Test

This notebook tests the Roboflow `InferenceHTTPClient` pipeline with the Roboflow Playground YOLOv4-tiny model.

YOLOv4-tiny is a general object detector and is not expected to reliably detect trees in aerial orthophotos.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from inference_sdk import InferenceHTTPClient
from PIL import Image, ImageDraw, ImageFont

In [ ]:
current_dir = Path.cwd().resolve()

if (current_dir / "notebooks").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

env_path = project_root / ".env"
load_dotenv(env_path)

api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise ValueError("ROBOFLOW_API_KEY is missing. Add it to .env before running this notebook.")

project_root

In [ ]:
# Roboflow model IDs usually use the form project_id/model_version_id.
# The Playground URL is /models/academia%20sinica/yolov4-tiny.
# Try the API-safe slug first, then the literal Playground path.
MODEL_IDS_TO_TRY = [
    "academia-sinica/yolov4-tiny",
    "academia sinica/yolov4-tiny",
]

image_path = project_root / "data" / "vienna_orthofoto_test.png"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"
predictions_path = project_root / "outputs" / "yolov4_tiny_predictions.json"
visualization_path = project_root / "outputs" / "yolov4_tiny_result.png"

if not image_path.exists():
    raise FileNotFoundError(f"Input image does not exist: {image_path}")

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing georeferencing metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

from backend.app.imagery.vienna_orthofoto import image_pixel_to_lonlat

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
bbox = metadata["bbox"]
image_width = metadata["image_width"]
image_height = metadata["image_height"]
default_lat = metadata["center"]["lat"]
default_lon = metadata["center"]["lon"]

predictions_path.parent.mkdir(parents=True, exist_ok=True)

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=api_key,
)

print(f"Models to try: {MODEL_IDS_TO_TRY}")
print(f"Image: {image_path}")
print(f"Default Vienna center: lat={default_lat}, lon={default_lon}")
print(f"Image CRS: {metadata['crs']}")
print(f"Image size: {image_width} x {image_height}")

In [ ]:
errors = []
result = None
used_model_id = None

for model_id in MODEL_IDS_TO_TRY:
    try:
        print(f"Trying model: {model_id}")
        result = client.infer(str(image_path), model_id=model_id)
        used_model_id = model_id
        break
    except Exception as exc:
        errors.append(f"{model_id}: {type(exc).__name__}: {exc}")
        print(f"Failed: {model_id} ({type(exc).__name__})")

if result is None:
    raise RuntimeError(
        "Roboflow hosted inference did not return a usable response for the Playground YOLOv4-tiny model. "
        "The current Roboflow Inference pre-trained alias list does not include YOLOv4-tiny, so this legacy "
        "Playground card may not be exposed as a stable InferenceHTTPClient endpoint.\n\n"
        + "\n".join(errors)
    )

result["experimental_model_id"] = used_model_id
result["input_metadata"] = {
    "source": metadata["source"],
    "center": metadata["center"],
    "bbox": metadata["bbox"],
    "bbox_lonlat": metadata["bbox_lonlat"],
    "crs": metadata["crs"],
    "image_width": image_width,
    "image_height": image_height,
}

predictions = result.get("predictions", [])
for prediction in predictions:
    lon, lat = image_pixel_to_lonlat(
        px=float(prediction["x"]),
        py=float(prediction["y"]),
        bbox=(bbox["min_x"], bbox["min_y"], bbox["max_x"], bbox["max_y"]),
        image_width=image_width,
        image_height=image_height,
    )
    prediction["lon"] = lon
    prediction["lat"] = lat

with predictions_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, indent=2)

print(f"Used model: {used_model_id}")
print(f"Saved raw predictions to: {predictions_path}")
print(f"Prediction count: {len(predictions)}")

In [ ]:
image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
font = ImageFont.load_default()

for prediction in predictions:
    x = float(prediction["x"])
    y = float(prediction["y"])
    width = float(prediction["width"])
    height = float(prediction["height"])

    left = x - width / 2
    top = y - height / 2
    right = x + width / 2
    bottom = y + height / 2

    class_name = prediction.get("class", "object")
    confidence = float(prediction.get("confidence", 0.0))
    label = f"{class_name} {confidence:.2f}"

    draw.rectangle((left, top, right, bottom), outline="lime", width=3)
    text_bbox = draw.textbbox((left, top), label, font=font)
    text_height = text_bbox[3] - text_bbox[1]
    text_y = max(0, top - text_height - 4)
    draw.rectangle((left, text_y, text_bbox[2] + 4, text_y + text_height + 4), fill="lime")
    draw.text((left + 2, text_y + 2), label, fill="black", font=font)

image.save(visualization_path)
print(f"Saved visualization to: {visualization_path}")

In [ ]:
if predictions:
    print("Detected classes and confidences:")
    for prediction in predictions:
        class_name = prediction.get("class", "object")
        confidence = float(prediction.get("confidence", 0.0))
        lon = prediction.get("lon")
        lat = prediction.get("lat")
        if lon is None or lat is None:
            print(f"- {class_name}: {confidence:.3f}")
        else:
            print(f"- {class_name}: {confidence:.3f} at lon={lon:.6f}, lat={lat:.6f}")
else:
    print("No detections returned by YOLOv4-tiny.")